## Feature Engineering: Smoothed Strike Rates for Pitchers, Umpires, Catchers and Batters 

The feature engineering process aimed to provide the predictive model with contextual insights on how the characteristics of different pitchers, umpires, catchers, and batters influence the likelihood of a pitch being a strike. Since each player (pitcher, batter, umpire, and catcher) has varying numbers of pitches associated with them, it is essential to create features that consider these differences without introducing data leakage.

To address this, smoothed strike rates were computed for the pitcher, umpire, catcher, and batter involved in each pitch. This method calculates each player's strike rate based on their past pitches up to that point, which prevents the model from using future information, thereby avoiding data leakage. 

The smoothed strike rate for each player (pitcher, umpire, catcher, or batter) is calculated using the following formula:
$$
\text{Smoothed Rate}_t = \frac{(m \cdot \text{Overall Strike Rate}) + (\text{Player Strike Rate}_{t-1} \cdot \text{Player Count}_{t-1})}{m + \text{Player Count}_{t-1}}
$$

Where:
- \( m \) is the smoothing parameter, which controls the balance between the overall strike rate and the player’s specific strike rate.
- **Overall Strike Rate** is the average strike rate across all pitches in the dataset.
- **Player Strike Rate** up to time \( t-1 \) is the player’s cumulative strike rate up to, but not including, time \( t \), calculated as:

  $$
  \text{Player Strike Rate}_{t-1} = \frac{\text{Number of Strikes by Player}_{t-1}}{\text{Total Pitches by Player}_{t-1}}
  $$

- **Player Count** up to time \( t-1 \) is the total number of pitches the player has participated in up to, but not including, time \( t \), denoted as \( \text{Player Count}_{t-1} \).

This formula ensures that the smoothed rate at each time \( t \) only depends on data up to \( t-1 \), preventing any look-ahead bias or data leakage.


The following code demonstrates this process:

**Importing Libraries**

In [40]:
import pandas as pd 
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import TimeSeriesSplit

**Importing Dataset & Calculating Overall Strike Rate**

In [41]:

df = pd.read_csv("C:\\Users\\Admin\\Downloads\\pitch_data (1).csv")
df = df[df['is_swing'] != 1]
overall_strike_rate = df['is_strike'].mean()


**Define Smoothing Parameters and Set Up Cross-Validation**

Set Range of 𝑚 Values: Various 𝑚 values are tested to determine the optimal level of smoothing. The parameter 𝑚 controls how much weight is given to the overall strike rate versus the individual player’s strike rate.

We test multiple values of 𝑚 to find the one that best balances the player-specific rate with the overall rate. Smaller 𝑚 values make the smoothed rate more sensitive to individual data, while larger values increase the influence of the overall strike rate.

In [42]:
# Define range of m values to test
m_values = [1, 5, 10, 20, 50, 100]
results = []


In [43]:
# Identify the entities to calculate smoothed strike rates for
entities = {
    'pitcherid': 'smoothed_pitcher_rate',
    'hp_umpid': 'smoothed_umpire_rate',
    'cid': 'smoothed_catcher_rate',
    'batterid': 'smoothed_batter_rate'
}


**Time Series Split for Cross-Validation**

A time series split is used for cross-validation to ensure that each fold only uses past data to make predictions about future data, maintaining the temporal sequence of evennts in the dataset 

In [44]:
# Time series split to avoid future data leakage
tscv = TimeSeriesSplit(n_splits=5) 

### Training Logistic Regression for Each \( m \) Value

For each \( m \) value, the logistic regression model is trained on smoothed strike rates calculated with that \( m \). This parameter \( m \) influences the degree of smoothing:

- **Low \( m \) values**: The model becomes more sensitive to individual data points, as it relies more heavily on player-specific data.
- **High \( m \) values**: The model becomes more generalized, as it places more weight on the overall strike rate, thereby reducing the influence of individual variances.

During training, logistic regression attempts to find the best coefficients for the smoothed strike rates of pitchers, umpires, catchers, and batters to predict the probability that a pitch is a strike is_strike = 1.

This process repeats across all training folds for each \( m \), so the model’s performance can be evaluated on unseen data.


In [45]:
#Iterate over different m values to find the optimal one for each entity
for m in m_values:
    log_loss_scores = []

    for train_index, test_index in tscv.split(df):
        train_data, test_data = df.iloc[train_index], df.iloc[test_index]
        
        # Initialize dictionaries to store smoothed strike rates for each entity
        smoothed_train_rates = {entity: [] for entity in entities}
        smoothed_test_rates = {entity: [] for entity in entities}
        counts = {entity: {} for entity in entities}
        strike_counts = {entity: {} for entity in entities}

        # Calculate smoothed strike rates incrementally for the training set
        for _, row in train_data.iterrows():
            for entity, feature_name in entities.items():
                entity_id = row[entity]
                
                # Calculate entity-specific strike rate
                entity_strike_rate = strike_counts[entity].get(entity_id, 0) / max(counts[entity].get(entity_id, 1), 1)
                smoothed_rate = (overall_strike_rate * m + entity_strike_rate * counts[entity].get(entity_id, 0)) / (m + counts[entity].get(entity_id, 0))
                
                # Append to the corresponding list for the training data
                smoothed_train_rates[entity].append(smoothed_rate)

                # Update counts for each entity
                counts[entity][entity_id] = counts[entity].get(entity_id, 0) + 1
                strike_counts[entity][entity_id] = strike_counts[entity].get(entity_id, 0) + row['is_strike']
        
        # Add smoothed rates to the training DataFrame
        for entity, feature_name in entities.items():
            train_data[feature_name] = smoothed_train_rates[entity]

        # Fit a logistic regression model using all smoothed rates
        model = LogisticRegression()
        model.fit(train_data[list(entities.values())], train_data['is_strike'])
    
        # Calculate smoothed rates for the test data using counts from training data
        for _, row in test_data.iterrows():
            for entity, feature_name in entities.items():
                entity_id = row[entity]
                entity_strike_rate = strike_counts[entity].get(entity_id, 0) / max(counts[entity].get(entity_id, 1), 1)
                smoothed_rate = (overall_strike_rate * m + entity_strike_rate * counts[entity].get(entity_id, 0)) / (m + counts[entity].get(entity_id, 0))
                smoothed_test_rates[entity].append(smoothed_rate)

        # Add smoothed rates to the test DataFrame
        for entity, feature_name in entities.items():
            test_data[feature_name] = smoothed_test_rates[entity]
            smoothed_test_rates[entity] = []  # Reset for the next fold

        # Make predictions and calculate log loss for the current fold
        predictions = model.predict_proba(test_data[list(entities.values())])[:, 1]
        score = log_loss(test_data['is_strike'], predictions)
        log_loss_scores.append(score)
        

    # Calculate the average log loss across all folds for the current m value
    average_log_loss = np.mean(log_loss_scores)
    results.append((m, average_log_loss))
    print(f"m={m}, Average Log Loss={average_log_loss}")



C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

m=1, Average Log Loss=0.6220207537391438


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

m=5, Average Log Loss=0.6215311860712478


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

...

C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

m=50, Average Log Loss=0.620912261755778


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[feature_name] = smoothed_train_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

m=100, Average Log Loss=0.6209683724011643


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[feature_name] = smoothed_test_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[feature_name] = smoothed_test_rates[entity]
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\3992355349.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[r

**Evaluating Model Performance with Log Loss**:
   - Once the model is trained, we calculate the smoothed strike rates for the test set using the same \( m \) value.
   - We then use the trained logistic regression model to predict the probability of each pitch being a strike in the test set. 
   
 **Log Loss Calculation**: 
   - Log loss is used to evaluate the accuracy of probability predictions. It penalizes incorrect predictions more heavily as the predicted probability diverges from the actual outcome.
   - Lower log loss values indicate better model performance. Here, it helps us identify the optimal \( m \) that minimizes the error across all test folds.


In [46]:
# Find the m with the lowest average log loss
optimal_m, best_score = min(results, key=lambda x: x[1])
print(f"\nOptimal m: {optimal_m}, with Log Loss: {best_score}")



Optimal m: 50, with Log Loss: 0.620912261755778


**Optimal value for the smooth rate features for Catchers, Umpires, Batters and Pitchers is $m$ = 50**
    
Smoothed Strike Rates are then calculated for Catchers, Umpires, Batters and Pitchers using $m$ = 50

In [47]:
# Recalculate smoothed strike rates for the entire dataset using the optimal m value
counts = {entity: {} for entity in entities}
strike_counts = {entity: {} for entity in entities}
smoothed_rates = {entity: [] for entity in entities}

for _, row in df.iterrows():
    for entity, feature_name in entities.items():
        entity_id = row[entity]
        entity_strike_rate = strike_counts[entity].get(entity_id, 0) / max(counts[entity].get(entity_id, 1), 1)
        smoothed_rate = (overall_strike_rate * optimal_m + entity_strike_rate * counts[entity].get(entity_id, 0)) / (optimal_m + counts[entity].get(entity_id, 0))
        smoothed_rates[entity].append(smoothed_rate)
        
        # Update counts
        counts[entity][entity_id] = counts[entity].get(entity_id, 0) + 1
        strike_counts[entity][entity_id] = strike_counts[entity].get(entity_id, 0) + row['is_strike']

# Append smoothed rates as new columns in the original dataframe
for entity, feature_name in entities.items():
    df[feature_name] = smoothed_rates[entity]

# Display the modified DataFrame with new smoothed columns
print(df.head())

   is_strike  is_swing  inning  is_bottom  balls  strikes  outs_before  \
0          1         0       1          0      0        0            0   
1          0         0       1          0      0        1            0   
3          0         0       1          0      0        0            0   
4          0         0       1          0      1        0            0   
5          1         0       1          0      2        0            0   

   is_lhp  is_lhb pitch_type  ...  plate_location_x  plate_location_z  \
0       0       0         FF  ...            -0.794             2.729   
1       0       0         FF  ...            -0.300             3.230   
3       0       0         SL  ...            -1.123             0.618   
4       0       0         FF  ...             0.692             3.932   
5       0       0         SL  ...            -0.927             2.727   

   rel_speed    spin_rate induced_vert_break horizontal_break  \
0  94.256378  2414.069843          18.494386       

## Feature Engineering: Smoothed Strike Rates for Umpire-Pitcher, Umpire-Catcher and Umpire-Batter Combinations 
Perform the same calculation as before however now look at the strike out rates of different combinations of Umpire-Pitcher, Umpire-Catcer

**Importance of Analyzing Strikeout Rates for Umpire-Pitcher, Umpire-Catcher, and Umpire-Batter Combinations**

1. **Umpire-Pitcher Combination**:
   - Helps identify how an umpire’s strike zone tendencies affect specific pitchers.
   - Highlights how a pitcher’s style (e.g., fastball-heavy, control) interacts with an umpire’s strike-calling consistency.
   - Provides insights into how umpires may adjust their strike zones based on who is pitching.

2. **Umpire-Catcher Combination**:
   - Reveals the influence of a catcher’s framing skills on an umpire’s strike-calling.
   - Identifies which umpire-catcher pairs tend to result in more strikeouts due to framing or communication.
   - Shows the dynamic between catchers and umpires that affects the pitcher’s performance.

3. **Umpire-Batter Combination**:
   - Uncovers how a specific umpire’s strike zone influences individual batters’ likelihood of striking out.
   - Highlights whether certain umpires are more lenient or strict with particular batters or batting styles.
   - Provides insights into how an umpire’s tendencies affect batters with different plate discipline approaches.


To calculate the smoothed strikeout rate for each combination of Umpire-Pitcher, Umpire-Catcher, and Umpire-Batter, we use the following formula:

$$
\text{Smoothed Strikeout Rate}_{t} = \frac{(m \cdot \text{Overall Strikeout Rate}) + (\text{Pair Strikeout Rate}_{t-1} \cdot \text{Pair Count}_{t-1})}{m + \text{Pair Count}_{t-1}}
$$

Where:
- **Overall Strikeout Rate** is the average strikeout rate across all pairs.
- **Pair Strikeout Rate** up to time \( t-1 \) is the cumulative strikeout rate for the pair up to, but not including, time \( t \), calculated as:

  $$
  \text{Pair Strikeout Rate}_{t-1} = \frac{\text{Number of Strikeouts by Pair}_{t-1}}{\text{Total Opportunities by Pair}_{t-1}}
  $$

- **Pair Count** up to time \( t-1 \) is the total number of opportunities (e.g., pitches) that the pair has been involved in up to, but not including, time \( t \).

This formula ensures that the smoothed strikeout rate at each time \( t \) only depends on past data, preventing any look-ahead bias or data leakage.


In [48]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import TimeSeriesSplit

# Initialize overall strike rate and define m values
overall_strike_rate = df['is_strike'].mean()
m_values = [1, 5, 10, 20, 50, 100, 200]
results = []

# Define pairs to calculate smoothed rates for
pairs = {
    'pitcher-umpire': ('pitcherid', 'hp_umpid', 'smoothed_pair_rateup'),
    'batter-umpire': ('batterid', 'hp_umpid', 'smoothed_pair_rateub'),
    'catcher-umpire': ('cid', 'hp_umpid', 'smoothed_pair_ratecu')
}

# Perform time series split for cross-validation
tscv = TimeSeriesSplit(n_splits=5)

# Function to calculate smoothed rates and evaluate log loss for a given pair
def calculate_smoothed_rates(pair, pair_name):
    entity1, entity2, rate_column = pairs[pair]
    best_m, best_log_loss = None, float('inf')
    
    for m in m_values:
        log_loss_scores = []
        
        # Time series split
        for train_index, test_index in tscv.split(df):
            train_data, test_data = df.iloc[train_index], df.iloc[test_index]
            
            # Initialize counts for the pair
            counts, strike_counts = {}, {}
            smoothed_rates_train, smoothed_rates_test = [], []

            # Training data smoothed rate calculation
            for _, row in train_data.iterrows():
                pair_key = (row[entity1], row[entity2])
                pair_strike_rate = strike_counts.get(pair_key, 0) / max(counts.get(pair_key, 1), 1)
                smoothed_rate = (overall_strike_rate * m + pair_strike_rate * counts.get(pair_key, 0)) / (m + counts.get(pair_key, 0))
                
                smoothed_rates_train.append(smoothed_rate)
                counts[pair_key] = counts.get(pair_key, 0) + 1
                strike_counts[pair_key] = strike_counts.get(pair_key, 0) + row['is_strike']
            
            # Add smoothed rates to training data
            train_data[rate_column] = smoothed_rates_train

            # Train logistic regression model
            model = LogisticRegression()
            model.fit(train_data[[rate_column]], train_data['is_strike'])
            
            # Test data smoothed rate calculation
            for _, row in test_data.iterrows():
                pair_key = (row[entity1], row[entity2])
                pair_strike_rate = strike_counts.get(pair_key, 0) / max(counts.get(pair_key, 1), 1)
                smoothed_rate = (overall_strike_rate * m + pair_strike_rate * counts.get(pair_key, 0)) / (m + counts.get(pair_key, 0))
                
                smoothed_rates_test.append(smoothed_rate)
            
            # Add smoothed rates to test data and evaluate
            test_data[rate_column] = smoothed_rates_test
            predictions = model.predict_proba(test_data[[rate_column]])[:, 1]
            log_loss_scores.append(log_loss(test_data['is_strike'], predictions))
        
        # Average log loss for current m
        avg_log_loss = np.mean(log_loss_scores)
        print(f"Pair: {pair_name}, m={m}, Average Log Loss={avg_log_loss}")

        # Store best m for current pair
        if avg_log_loss < best_log_loss:
            best_m, best_log_loss = m, avg_log_loss

    print(f"\nOptimal m for {pair_name}: {best_m} with Log Loss: {best_log_loss}")

    # Final calculation for optimal m
    counts, strike_counts, smoothed_rates_final = {}, {}, []
    for _, row in df.iterrows():
        pair_key = (row[entity1], row[entity2])
        pair_strike_rate = strike_counts.get(pair_key, 0) / max(counts.get(pair_key, 1), 1)
        smoothed_rate = (overall_strike_rate * best_m + pair_strike_rate * counts.get(pair_key, 0)) / (best_m + counts.get(pair_key, 0))
        
        smoothed_rates_final.append(smoothed_rate)
        counts[pair_key] = counts.get(pair_key, 0) + 1
        strike_counts[pair_key] = strike_counts.get(pair_key, 0) + row['is_strike']

    # Append the final smoothed rates to the dataset
    df[rate_column] = smoothed_rates_final

# Run the smoothed rate calculation for each defined pair
for pair_name in pairs:
    calculate_smoothed_rates(pair_name, pair_name)

# Display the updated data with new smoothed rate columns
print(df.head())


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[rate_column] = smoothed_rates_train
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[rate_column] = smoothed_rates_test
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

Pair: pitcher-umpire, m=1, Average Log Loss=0.6219691025811059


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[rate_column] = smoothed_rates_train
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[rate_column] = smoothed_rates_test
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

Pair: pitcher-umpire, m=5, Average Log Loss=0.62197605194804


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[rate_column] = smoothed_rates_train
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[rate_column] = smoothed_rates_test
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

...

Pair: catcher-umpire, m=50, Average Log Loss=0.6219815504760755


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[rate_column] = smoothed_rates_train
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[rate_column] = smoothed_rates_test
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

Pair: catcher-umpire, m=100, Average Log Loss=0.6219661922920773


C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[rate_column] = smoothed_rates_train
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[rate_column] = smoothed_rates_test
C:\Users\Admin\AppData\Local\Temp\ipykernel_27612\936520194.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

Pair: catcher-umpire, m=200, Average Log Loss=0.6219658126165739

Optimal m for catcher-umpire: 200 with Log Loss: 0.6219658126165739
   is_strike  is_swing  inning  is_bottom  balls  strikes  outs_before  \
0          1         0       1          0      0        0            0   
1          0         0       1          0      0        1            0   
3          0         0       1          0      0        0            0   
4          0         0       1          0      1        0            0   
5          1         0       1          0      2        0            0   

   is_lhp  is_lhb pitch_type  ...    spin_rate  induced_vert_break  \
0       0       0         FF  ...  2414.069843           18.494386   
1       0       0         FF  ...  2444.899677           16.832286   
3       0       0         SL  ...  2619.399677            5.961645   
4       0       0         FF  ...  2400.939716           18.684886   
5       0       0         SL  ...  2509.159687            8.828125   



In [56]:
# Fill in the missing values in spinrate column with the mean value for the same pitch type
df['spin_rate'] = df.groupby('pitch_type')['spin_rate'].transform(lambda x: x.fillna(x.mean()))
# Filter for non-swung pitches
df = df[df['is_swing'] == 0]

## Dummification of Inning & Pitch Type 

Dummification of both categories is key, although they are numerical values we want the model treating these factors as categorical 

In [57]:

# Dummify both pitch_type and inning columns at the same time
data = pd.get_dummies(df, columns=['pitch_type', 'inning'])


### Calculating the Center of the Strike Zone and Distance for Each Pitch (Euclidean Distance)

The code calculates the center of the strike zone based on pitches that were called strikes without a swing. This is done by:

1. **Filtering Called Strikes**:
   - The dataset is filtered for pitches where the batter did not swing (`is_swing == 0`) and the pitch was called a strike (`is_strike == 1`).

2. **Determining Strike Zone Bounds**:
   - The 2.5th and 97.5th percentiles of the `plate_location_x` and `plate_location_z` coordinates are used to define the bounds of the strike zone. These bounds help capture the general limits where pitches are most often called strikes.

   $$
   x_{\text{bounds}} = \left[ \text{Percentile}_{2.5}(\text{plate\_location\_x}), \text{Percentile}_{97.5}(\text{plate\_location\_x}) \right]
   $$
   $$
   z_{\text{bounds}} = \left[ \text{Percentile}_{2.5}(\text{plate\_location\_z}), \text{Percentile}_{97.5}(\text{plate\_location\_z}) \right]
   $$

3. **Calculating the Center of the Strike Zone**:
   - The center of the strike zone is then computed as the midpoint of the x and z bounds.

   $$
   x_{\text{center}} = \frac{x_{\text{bounds}_{0}} + x_{\text{bounds}_{1}}}{2}
   $$
   $$
   z_{\text{center}} = \frac{z_{\text{bounds}_{0}} + z_{\text{bounds}_{1}}}{2}
   $$

4. **Calculating Distance from the Center**:
   - For each pitch, the Euclidean distance from the calculated center of the strike zone is computed and added as a new feature, `distance_from_center`, which represents how far each pitch is from the strike zone center.
   
   $$
   \text{Distance from Center} = \sqrt{(x - x_{\text{center}})^2 + (z - z_{\text{center}})^2}
   $$


**Note**: This calculation involves data from future events (as it uses all pitches to define the strike zone), which could introduce a bias if included in a predictive model. Therefore, it was excluded from features used for training to avoid data leakage. However it is interesting idea that could be implemented in future model developments 

In [50]:
# Filter for non-swung pitches that were called strikes to calculate the strike zone bounds
called_strikes = data[(data['is_swing'] == 0) & (data['is_strike'] == 1)]

# Calculate the center of the strike zone
x_bounds = np.percentile(called_strikes['plate_location_x'], [2.5, 97.5])
z_bounds = np.percentile(called_strikes['plate_location_z'], [2.5, 97.5])

x_center = (x_bounds[0] + x_bounds[1]) / 2
z_center = (z_bounds[0] + z_bounds[1]) / 2

print("Center of Strike Zone (X, Z):", (x_center, z_center))

# Calculate the Euclidean distance from the center of the strike zone for each pitch
data['distance_from_center'] = np.sqrt((data['plate_location_x'] - x_center)**2 + 
                                       (data['plate_location_z'] - z_center)**2)

Center of Strike Zone (X, Z): (-0.005199999999999982, 2.424)


In [58]:

# export the data to a new csv file
data.to_csv('C:\\Users\\Admin\\Downloads\\pitch_data_FeatureEngineered(3).csv', index=False)